<a href="https://colab.research.google.com/github/lovkush7/NLP--P/blob/main/spam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
path ="/content/drive/MyDrive/sms_spam_detection_dataset.csv"

In [14]:
import pandas as pd
df = pd.read_csv(path)

In [15]:
df.head()

,sms_text,label
0,I have shared the project files in the group.,ham
1,your package is on hold. pay the small deliver...,spam
2,"hey, are we still meeting after class today?",ham
3,Your appointment is confirmed for Monday.,ham
4,We are having a family dinner on Saturday.,ham


In [16]:
df.duplicated().sum()

np.int64(132)

In [17]:
df.drop_duplicates(inplace=True)

In [18]:
df.shape

(118, 2)

In [19]:
df.isnull().sum()

,0
sms_text,0
label,0


In [20]:
df['label'].value_counts()

,count
label,
ham,59
spam,59


In [21]:
import re
def  clean_text(text):
  text = text.lower()
      # Remove URLs
    # Remove URLs
  text = re.sub(r"http\S+|www\S+", "", text)

    # Remove HTML tags
  text = re.sub(r"<.*?>", "", text)

    # Remove numbers
  text = re.sub(r"\d+", "", text)

    # Remove punctuation and special characters
  text = re.sub(r"[^a-zA-Z\s]", "", text)

    # Remove extra spaces
  text = re.sub(r"\s+", " ", text).strip()

  return text

  df['sms_text'] = df['sms_text'].apply(clean_text)

In [22]:
df["sms_text"] = df["sms_text"].fillna("")

df = df[df["sms_text"].str.strip() != ""]

df = df.reset_index(drop=True)

In [23]:
import nltk
from nltk.tokenize import word_tokenize
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

df["token"] = df['sms_text'].apply(word_tokenize)
print(df[['sms_text' , 'token']].head())

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


                                            sms_text  \
0      I have shared the project files in the group.   
1  your package is on hold. pay the small deliver...   
2       hey, are we still meeting after class today?   
3          Your appointment is confirmed for Monday.   
4         We are having a family dinner on Saturday.   

                                               token  
0  [I, have, shared, the, project, files, in, the...  
1  [your, package, is, on, hold, ., pay, the, sma...  
2  [hey, ,, are, we, still, meeting, after, class...  
3  [Your, appointment, is, confirmed, for, Monday...  
4  [We, are, having, a, family, dinner, on, Satur...  


In [24]:
from nltk.corpus import stopwords
stop_word = set(stopwords.words('english'))

In [26]:
nltk.download('wordnet')
nltk.download('omw-1.4')

from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

df['lemmatizer'] = df['token'].apply(
    lambda words: [
        lemmatizer.lemmatize(word)
        for word in words
    ]


)

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [29]:
df['final_text'] = df['lemmatizer'].apply(
    lambda words: " ".join(words)
)

In [30]:
print(df[['final_text', 'lemmatizer']].head())

                                          final_text  \
0      I have shared the project file in the group .   
1  your package is on hold . pay the small delive...   
2     hey , are we still meeting after class today ?   
3         Your appointment is confirmed for Monday .   
4        We are having a family dinner on Saturday .   

                                          lemmatizer  
0  [I, have, shared, the, project, file, in, the,...  
1  [your, package, is, on, hold, ., pay, the, sma...  
2  [hey, ,, are, we, still, meeting, after, class...  
3  [Your, appointment, is, confirmed, for, Monday...  
4  [We, are, having, a, family, dinner, on, Satur...  


In [32]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

In [34]:
X = df['final_text']
y = df['label']

In [37]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)



In [38]:
bow = CountVectorizer()
X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.fit_transform(X_test)

In [42]:
tfidf = TfidfVectorizer()
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [43]:
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
nb_pred = nb_model.predict(X_test_tfidf)

In [44]:
svm_model = LinearSVC()
svm_model.fit(X_train_tfidf, y_train)
svm_pred = svm_model.predict(X_test_tfidf)


In [45]:
lr_model = LogisticRegression()
lr_model.fit(X_train_tfidf, y_train)
lr_pred = lr_model.predict(X_test_tfidf)

In [46]:
models = {
    "Naive Bayes": nb_pred,
    "SVM": svm_pred,
    "Logistic Regression": lr_pred
}

for name, prediction in models.items():

    print("\nModel:", name)

    print("Accuracy:",
          accuracy_score(y_test, prediction))

    print("\nClassification Report:")

    print(classification_report(y_test, prediction))


Model: Naive Bayes
Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

         ham       1.00      1.00      1.00        12
        spam       1.00      1.00      1.00        12

    accuracy                           1.00        24
   macro avg       1.00      1.00      1.00        24
weighted avg       1.00      1.00      1.00        24


Model: SVM
Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

         ham       1.00      1.00      1.00        12
        spam       1.00      1.00      1.00        12

    accuracy                           1.00        24
   macro avg       1.00      1.00      1.00        24
weighted avg       1.00      1.00      1.00        24


Model: Logistic Regression
Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

         ham       1.00      1.00      1.00        12
        spam       1.00      1.00      1.00        12

    accu

In [47]:
results = []

for name, prediction in models.items():

    accuracy = accuracy_score(y_test, prediction)

    results.append({
        "Model": name,
        "Accuracy": accuracy
    })

result_df = pd.DataFrame(results)

print(result_df.sort_values(
    by="Accuracy",
    ascending=False
))

                 Model  Accuracy
0          Naive Bayes       1.0
1                  SVM       1.0
2  Logistic Regression       1.0


In [49]:
new_review = ["please give me your bank details"]

new_vector = tfidf.transform(new_review)

prediction = svm_model.predict(new_vector)

print("Predicted Sentiment:", prediction[0])

Predicted Sentiment: ham
